# Cinemática Diferencial e Jacobiana Geométrica
## Aplicação: Denso VP-6242 — Parâmetros DH Padrão

**Robótica — Nível Graduação**

Este notebook implementa passo a passo:
1. Parametrização DH padrão do Denso VP-6242  
2. Cinemática direta (matrizes homogêneas)  
3. Jacobiana Geométrica — dedução e cálculo  
4. Análise de singularidades  
5. Controle de velocidade cartesiana (pseudo-inversa e DLS)  
6. Simulação de trajetória no espaço da tarefa  

---
## 0. Dependências

In [ ]:
import numpy as np
import matplotlib
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D
# from mpl_toolkits.mplot3d.art3d import Line3DCollection
# import matplotlib.animation as animation
# from IPython.display import HTML, display
#
# np.set_printoptions(precision=4, suppress=True)
# plt.rcParams.update({
#     'figure.dpi': 110,
#     'axes.titlesize': 13,
#     'axes.labelsize': 11,
#     'font.family': 'DejaVu Sans',
# })
# print('Dependências carregadas.')

---
## 1. Parâmetros DH — Denso VP-6242

Convenção DH **padrão** (Craig). A tabela abaixo define os quatro parâmetros de cada junta:

| Junta | $a_i$ (mm) | $\alpha_i$ (°) | $d_i$ (mm) | $\theta_i$ |
|:-----:|:----------:|:--------------:|:----------:|:----------:|
|   1   |     0      |     −90        |   263.5    |   $q_1$    |
|   2   |    210     |       0        |     0      |   $q_2$    |
|   3   |     75     |     −90        |     0      |   $q_3$    |
|   4   |     0      |      90        |   222.5    |   $q_4$    |
|   5   |     0      |     −90        |     0      |   $q_5$    |
|   6   |     0      |       0        |    36.5    |   $q_6$    |

Todas as juntas são de **revolução** → $\theta_i$ é a variável, demais parâmetros são constantes.

In [ ]:
# Parâmetros DH padrão: [a (mm), alpha (rad), d (mm), theta_offset (rad)]
# theta_offset = 0 para todas as juntas deste robô
DH = np.array([
    [  0.0,  np.radians(-90),  263.5,  0.0],   # junta 1
    [210.0,  np.radians(  0),    0.0,  0.0],   # junta 2
    [ 75.0,  np.radians(-90),    0.0,  0.0],   # junta 3
    [  0.0,  np.radians( 90),  222.5,  0.0],   # junta 4
    [  0.0,  np.radians(-90),    0.0,  0.0],   # junta 5
    [  0.0,  np.radians(  0),   36.5,  0.0],   # junta 6
])

n = len(DH)   # número de juntas
print(f'Robô: Denso VP-6242  |  {n} juntas de revolução')
print(f'Parâmetros DH (a[mm], alpha[rad], d[mm], theta_offset[rad]):')
print(DH)

---
## 2. Cinemática Direta

A matriz de transformação homogênea entre frames consecutivos é dada por:

$$
{}^{i-1}T_i = \begin{bmatrix}
c\theta_i & -s\theta_i c\alpha_i &  s\theta_i s\alpha_i & a_i c\theta_i \\
s\theta_i &  c\theta_i c\alpha_i & -c\theta_i s\alpha_i & a_i s\theta_i \\
0         &  s\alpha_i           &  c\alpha_i           & d_i           \\
0         &  0                   &  0                   & 1
\end{bmatrix}
$$

A cinemática direta completa é: ${}^0T_6 = {}^0T_1 \cdot {}^1T_2 \cdots {}^5T_6$

In [ ]:
def dh_matrix(a, alpha, d, theta):
    """Matriz DH padrão ^{i-1}T_i para uma junta de revolução."""
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ ct, -st*ca,  st*sa,  a*ct],
        [ st,  ct*ca, -ct*sa,  a*st],
        [  0,     sa,     ca,     d],
        [  0,      0,      0,     1],
    ])

def forward_kinematics(q, dh=DH):
    """
    Cinemática direta completa.

    Parâmetros
    ----------
    q   : array (6,) — ângulos das juntas em radianos
    dh  : array (6,4) — tabela DH padrão

    Retorna
    -------
    T_list : list de 7 matrizes (4×4) — T[0]=I, T[1]=^0T_1, ..., T[6]=^0T_6
    """
    T = np.eye(4)
    T_list = [T.copy()]
    for i in range(len(dh)):
        a, alpha, d, theta_off = dh[i]
        Ti = dh_matrix(a, alpha, d, q[i] + theta_off)
        T = T @ Ti
        T_list.append(T.copy())
    return T_list

# ----- Teste: configuração zero -----
q0 = np.zeros(6)
T_list = forward_kinematics(q0)

T06 = T_list[-1]
print('Configuração zero  q = [0, 0, 0, 0, 0, 0] rad')
print('\n^0T_6 =')
print(T06)
print(f'\nPosição do efetuador: p = {T06[:3,3].round(2)} mm')

In [ ]:
# ----- Teste: configuração arbitrária -----
q_test = np.radians([30, -45, 60, 0, 45, -30])
T_test = forward_kinematics(q_test)
p_test = T_test[-1][:3, 3]
R_test = T_test[-1][:3, :3]

print('Configuração q = [30°, -45°, 60°, 0°, 45°, -30°]')
print(f'\nPosição do efetuador:  p = {p_test.round(2)} mm')
print(f'\nOrientação (matriz R):')
print(R_test.round(4))

---
## 3. Jacobiana Geométrica

Para um robô com $n$ juntas de **revolução**, cada coluna $J_i$ é:

$$
J_i = \begin{bmatrix} J_{v,i} \\ J_{\omega,i} \end{bmatrix}
= \begin{bmatrix} \hat{z}_{i-1} \times (\mathbf{p}_e - \mathbf{p}_{i-1}) \\ \hat{z}_{i-1} \end{bmatrix}
$$

onde:
- $\hat{z}_{i-1}$ = 3ª coluna de ${}^0R_{i-1}$ (eixo de rotação da junta $i$ no frame base)  
- $\mathbf{p}_{i-1}$ = 4ª coluna de ${}^0T_{i-1}$ (origem do frame $i-1$ no frame base)  
- $\mathbf{p}_e$ = posição do efetuador = 4ª coluna de ${}^0T_6$

In [ ]:
def geometric_jacobian(q, dh=DH):
    """
    Jacobiana Geométrica (6 x n) para robô com n juntas de revolução.

    Parâmetros
    ----------
    q   : array (n,) — ângulos das juntas
    dh  : tabela DH

    Retorna
    -------
    J   : array (6, n)
    """
    n = len(q)
    T_list = forward_kinematics(q, dh)

    # posição do efetuador
    pe = T_list[-1][:3, 3]

    J = np.zeros((6, n))
    for i in range(n):
        # eixo z do frame i-1 no frame base (3ª coluna da submatriz de rotação)
        z_prev = T_list[i][:3, 2]          # T_list[i] = ^0T_{i-1} (frames 0..i-1)
        # origem do frame i-1 no frame base
        p_prev = T_list[i][:3, 3]

        # coluna de velocidade linear: z x (pe - p_{i-1})
        J[:3, i] = np.cross(z_prev, pe - p_prev)
        # coluna de velocidade angular: z_{i-1}
        J[3:, i] = z_prev

    return J

# ----- Teste q = 0 -----
J0 = geometric_jacobian(q0)
print('Jacobiana Geométrica J(q=0)  [6 x 6]  (mm/rad e rad/rad):')
print(J0.round(2))
print(f'\nPosto de J: {np.linalg.matrix_rank(J0)}')

In [ ]:
# ----- Visualização das colunas -----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

J_test = geometric_jacobian(q_test)

# Parte linear (linhas 0-2)
im0 = axes[0].imshow(J_test[:3, :], cmap='RdBu', aspect='auto',
                     vmin=-np.max(np.abs(J_test[:3,:])),
                     vmax= np.max(np.abs(J_test[:3,:])))
axes[0].set_title('Jv(q)  — parte linear (3×6)')
axes[0].set_xticks(range(6)); axes[0].set_xticklabels([f'J{i+1}' for i in range(6)])
axes[0].set_yticks(range(3)); axes[0].set_yticklabels(['vx','vy','vz'])
for r in range(3):
    for c in range(6):
        axes[0].text(c, r, f'{J_test[r,c]:.1f}', ha='center', va='center', fontsize=8)
plt.colorbar(im0, ax=axes[0])

# Parte angular (linhas 3-5)
im1 = axes[1].imshow(J_test[3:, :], cmap='RdBu', aspect='auto',
                     vmin=-1.1, vmax=1.1)
axes[1].set_title('Jω(q)  — parte angular (3×6)')
axes[1].set_xticks(range(6)); axes[1].set_xticklabels([f'J{i+1}' for i in range(6)])
axes[1].set_yticks(range(3)); axes[1].set_yticklabels(['ωx','ωy','ωz'])
for r in range(3):
    for c in range(6):
        axes[1].text(c, r, f'{J_test[3+r,c]:.3f}', ha='center', va='center', fontsize=8)
plt.colorbar(im1, ax=axes[1])

fig.suptitle(f'Jacobiana Geométrica — q = [30°,-45°,60°,0°,45°,-30°]', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Análise de Singularidades

A configuração $\mathbf{q}$ é singular quando:
$$\det(J J^T) = 0 \quad \Leftrightarrow \quad \text{rank}(J) < 6$$

Uma medida contínua é a **manipulabilidade** de Yoshikawa:
$$w(\mathbf{q}) = \sqrt{\det(J J^T)}$$

- $w > 0$: configuração regular  
- $w \to 0$: aproximando singularidade  
- $w = 0$: singularidade — alguma direção no espaço cartesiano é inatingível

Tipos no Denso VP-6242:
- **Ombro**: efetuador sobre o eixo $z_0$
- **Cotovelo**: $q_3 = 0°$ ou $\pm 180°$
- **Pulso**: $q_5 = 0°$ (eixos de J4 e J6 colineares)

In [ ]:
# Configuração base: q5 varia, demais fixos
q5_range = np.linspace(-np.pi, np.pi, 360)
q_scan = np.array([0.0, np.radians(-30), np.radians(45), np.radians(10), 0.0, np.radians(30)])

w_vals = []
for q5 in q5_range:
    q_scan[4] = q5
    w_vals.append(manipulability(q_scan))

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(np.degrees(q5_range), w_vals, color='#1E5FA6', lw=2)
ax.axvline(-180, color='#DC2626', ls='--', lw=1.5, label='q5=+/-180 graus (singularidade de pulso)')
ax.axvline( 180, color='#DC2626', ls='--', lw=1.5)
ax.set_xlabel('q5 (graus)')
ax.set_ylabel('w(q)  [manipulabilidade]')
ax.set_title('Manipulabilidade em funcao de q5 - singularidade de pulso')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

q_scan[4] = -np.pi
print(f'w(q5=-180 graus) = {manipulability(q_scan):.2e}  -> SINGULAR')
q_scan[4] = np.radians(45)
print(f'w(q5=45 graus)   = {manipulability(q_scan):.2e}')
q_scan[4] = np.radians(90)
print(f'w(q5=90 graus)   = {manipulability(q_scan):.2e}')

In [ ]:
# ---- Varredura 2D: q2 × q3 (mapa de manipulabilidade) ----
q2_range = np.linspace(-np.pi/2, np.pi/2, 80)
q3_range = np.linspace(-np.pi/2, np.pi/2, 80)
W = np.zeros((len(q3_range), len(q2_range)))

q_base = np.array([0.0, 0.0, 0.0, 0.0, np.radians(45), 0.0])
for i, q3 in enumerate(q3_range):
    for j, q2 in enumerate(q2_range):
        q_base[1] = q2; q_base[2] = q3
        W[i, j] = manipulability(q_base)

fig, ax = plt.subplots(figsize=(7, 5.5))
c = ax.contourf(np.degrees(q2_range), np.degrees(q3_range), W,
                levels=30, cmap='RdYlGn')
ax.contour(np.degrees(q2_range), np.degrees(q3_range), W,
           levels=[0], colors='red', linewidths=2)
plt.colorbar(c, ax=ax, label='w(q) [manipulabilidade]')
ax.set_xlabel('q2 (graus)'); ax.set_ylabel('q3 (graus)')
ax.set_title('Mapa de Manipulabilidade — plano (q2, q3)\n(linha vermelha = singularidade)')
ax.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()

---
## 5. Inversão Diferencial: Pseudo-Inversa e DLS

Dada a velocidade cartesiana desejada $\dot{\mathbf{x}}_d \in \mathbb{R}^6$, calcula-se $\dot{\mathbf{q}}$:

**Pseudo-inversa de Moore-Penrose:**
$$\dot{\mathbf{q}} = J^+(\mathbf{q})\,\dot{\mathbf{x}}_d, \quad J^+ = J^T(JJ^T)^{-1}$$

**Damped Least Squares (DLS)** — robusto perto de singularidades:
$$\dot{\mathbf{q}} = J^T(JJ^T + \lambda^2 I)^{-1}\dot{\mathbf{x}}_d$$

O parâmetro $\lambda > 0$ limita a amplificação das velocidades de junta.

In [ ]:
def pseudoinverse(J, rcond=1e-6):
    """Pseudo-inversa de Moore-Penrose via SVD."""
    return np.linalg.pinv(J, rcond=rcond)

def dls_inverse(J, lam=0.05):
    """
    Damped Least Squares: J^T (J J^T + lam^2 I)^{-1}
    lam : fator de amortecimento (ex: 0.01 a 0.1)
    """
    m = J.shape[0]
    return J.T @ np.linalg.inv(J @ J.T + lam**2 * np.eye(m))

def qdot_from_xdot(q, xdot_d, method='dls', lam=0.05, dh=DH):
    """
    Calcula velocidades de junta q_dot a partir da velocidade cartesiana desejada xdot_d.

    method : 'pinv' (pseudo-inversa) | 'dls' (amortecida)
    """
    J = geometric_jacobian(q, dh)
    if method == 'pinv':
        return pseudoinverse(J) @ xdot_d
    else:
        return dls_inverse(J, lam) @ xdot_d

# ---- Comparação pinv vs DLS perto de singularidade ----
lam_values = [0.001, 0.01, 0.05, 0.1, 0.5]
xdot_d = np.array([10., 0., 5., 0., 0., 0.])   # 10 mm/s em x, 5 mm/s em z

q_near_sing = np.array([0., np.radians(-45), 0., 0., 0., 0.])   # q3=0 → sing. de cotovelo
q_regular   = np.array([0., np.radians(-45), np.radians(60), 0., np.radians(90), 0.])

print(f'Manipulabilidade (singular):  w = {manipulability(q_near_sing):.4f}')
print(f'Manipulabilidade (regular):   w = {manipulability(q_regular):.4f}')

print('\n--- Norma de q_dot perto de singularidade ---')
print(f'{"lambda":>10}  {"||q_dot|| (DLS)":>20}  {"||q_dot|| (pinv)":>20}')
qdot_pinv = qdot_from_xdot(q_near_sing, xdot_d, method='pinv')
for lam in lam_values:
    qdot_dls = qdot_from_xdot(q_near_sing, xdot_d, method='dls', lam=lam)
    print(f'{lam:>10.3f}  {np.linalg.norm(qdot_dls):>20.4f}  {np.linalg.norm(qdot_pinv):>20.4f}')

In [ ]:
# ---- Gráfico: norma de q_dot vs lambda ----
lam_sweep = np.logspace(-3, 0, 200)
norm_sing = [np.linalg.norm(qdot_from_xdot(q_near_sing, xdot_d, 'dls', l)) for l in lam_sweep]
norm_reg  = [np.linalg.norm(qdot_from_xdot(q_regular,   xdot_d, 'dls', l)) for l in lam_sweep]

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(lam_sweep, norm_sing, '#DC2626', lw=2, label='Perto de singularidade (w≈0)')
ax.loglog(lam_sweep, norm_reg,  '#1E5FA6', lw=2, label='Configuração regular')
ax.axvline(0.05, color='gray', ls='--', alpha=0.7, label='λ = 0.05 (típico)')
ax.set_xlabel('λ (fator DLS)')
ax.set_ylabel('||q̇||  (rad/s)')
ax.set_title('Efeito do Amortecimento DLS na Norma de q̇')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

---
## 6. Visualização 3D do Robô

Função para desenhar o Denso VP-6242 em qualquer configuração.

In [ ]:
def plot_robot(q, ax=None, color='#1E5FA6', alpha=1.0, label=None,
               show_frames=False, frame_scale=50):
    """
    Desenha o robô Denso VP-6242 em 3D dado q (radianos).
    Retorna o eixo matplotlib 3D.
    """
    T_list = forward_kinematics(q)
    pts = np.array([T[:3, 3] for T in T_list])   # (7, 3)

    if ax is None:
        fig = plt.figure(figsize=(6, 6))
        ax = fig.add_subplot(111, projection='3d')

    # Links
    ax.plot(pts[:, 0], pts[:, 1], pts[:, 2],
            '-o', color=color, lw=4, ms=8, alpha=alpha,
            label=label, solid_capstyle='round')

    # Efetuador
    pe = pts[-1]
    ax.scatter(*pe, color='#DC2626', s=120, zorder=5)

    # Frames (opcional)
    if show_frames:
        for i, T in enumerate(T_list):
            o = T[:3, 3]
            for j, c in enumerate(['r', 'g', 'b']):
                d = T[:3, j] * frame_scale
                ax.quiver(*o, *d, color=c, arrow_length_ratio=0.3, lw=1.5)

    return ax

fig = plt.figure(figsize=(12, 5))
configs = [
    (np.zeros(6),                                     'q = 0  (extensão)'),
    (np.radians([0, -45, 90, 0, 45, 0]),              'q = [0,-45,90,0,45,0]°'),
    (np.radians([45, -30, 60, -20, 70, 30]),          'q = [45,-30,60,-20,70,30]°'),
]
colors = ['#1E5FA6', '#16A34A', '#9333EA']

for k, (q_k, title) in enumerate(configs):
    ax = fig.add_subplot(1, 3, k+1, projection='3d')
    plot_robot(q_k, ax=ax, color=colors[k], show_frames=True, frame_scale=40)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('X (mm)'); ax.set_ylabel('Y (mm)'); ax.set_zlabel('Z (mm)')
    ax.set_xlim(-500, 500); ax.set_ylim(-500, 500); ax.set_zlim(0, 700)
    ax.view_init(elev=20, azim=45)

fig.suptitle('Denso VP-6242 — Visualização 3D em diferentes configurações',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Simulação de Trajetória Cartesiana

Controle por velocidade: integra $\dot{\mathbf{q}} = J^+\dot{\mathbf{x}}_d$ para seguir uma trajetória circular no plano XZ.

In [ ]:
def simulate_cartesian_trajectory(q0, xdot_func, dt=0.01, T=2.0,
                                   method='dls', lam=0.05):
    """
    Integra o controle por velocidade cartesiana.

    Parâmetros
    ----------
    q0        : configuração inicial (rad)
    xdot_func : função xdot_func(t) → array (6,)
    dt        : passo de integração (s)
    T         : duração total (s)
    method    : 'dls' ou 'pinv'

    Retorna
    -------
    q_traj    : (N, 6) — trajetória no espaço de juntas
    p_traj    : (N, 3) — trajetória cartesiana do efetuador
    t_arr     : (N,)   — vetor de tempo
    w_traj    : (N,)   — manipulabilidade ao longo da trajetória
    """
    t_arr  = np.arange(0, T, dt)
    q_traj = np.zeros((len(t_arr), 6))
    p_traj = np.zeros((len(t_arr), 3))
    w_traj = np.zeros(len(t_arr))

    q = q0.copy()
    for k, t in enumerate(t_arr):
        T_list = forward_kinematics(q)
        p_traj[k] = T_list[-1][:3, 3]
        q_traj[k] = q
        w_traj[k] = manipulability(q)

        xdot = xdot_func(t)
        qdot = qdot_from_xdot(q, xdot, method=method, lam=lam)
        q = q + qdot * dt

    return q_traj, p_traj, t_arr, w_traj

# ---- Trajetória circular no plano XY ----
radius = 80.0      # mm
omega  = np.pi     # rad/s  →  período = 2s

def xdot_circle(t):
    """Velocidade tangencial de círculo no plano XY."""
    vx = -radius * omega * np.sin(omega * t)
    vy =  radius * omega * np.cos(omega * t)
    return np.array([vx, vy, 0., 0., 0., 0.])

q_init = np.radians([0, -30, 60, 0, 60, 0])

q_traj, p_traj, t_arr, w_traj = simulate_cartesian_trajectory(
    q_init, xdot_circle, dt=0.005, T=2.0, method='dls', lam=0.05
)

print(f'Simulação concluída: {len(t_arr)} passos')
print(f'Posição inicial:  {p_traj[0].round(1)} mm')
print(f'Posição final:    {p_traj[-1].round(1)} mm')
print(f'Erro de fechamento: {np.linalg.norm(p_traj[-1]-p_traj[0]):.2f} mm')

In [ ]:
fig = plt.figure(figsize=(14, 5))

# ---- Trajetória 3D ----
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.plot(p_traj[:,0], p_traj[:,1], p_traj[:,2],
         color='#DC2626', lw=2, label='Trajetória EEF')
ax1.scatter(*p_traj[0],  color='green', s=80, zorder=5, label='Início')
ax1.scatter(*p_traj[-1], color='blue',  s=80, zorder=5, label='Fim')
# robô na posição inicial (translúcido)
plot_robot(q_init, ax=ax1, color='#1E5FA6', alpha=0.4)
ax1.set_title('Trajetória no Espaço 3D')
ax1.set_xlabel('X (mm)'); ax1.set_ylabel('Y (mm)'); ax1.set_zlabel('Z (mm)')
ax1.legend(fontsize=8); ax1.view_init(elev=25, azim=40)

# ---- Ângulos das juntas ao longo do tempo ----
ax2 = fig.add_subplot(1, 3, 2)
for i in range(6):
    ax2.plot(t_arr, np.degrees(q_traj[:,i]), label=f'q{i+1}', lw=1.5)
ax2.set_xlabel('Tempo (s)'); ax2.set_ylabel('Ângulo (°)')
ax2.set_title('Ângulos das Juntas q(t)')
ax2.legend(fontsize=8, ncol=2); ax2.grid(True, alpha=0.3)

# ---- Manipulabilidade ao longo do tempo ----
ax3 = fig.add_subplot(1, 3, 3)
ax3.plot(t_arr, w_traj, color='#1E5FA6', lw=2)
ax3.axhline(y=np.mean(w_traj), color='gray', ls='--',
            label=f'Média = {np.mean(w_traj):.2e}')
ax3.fill_between(t_arr, 0, w_traj, alpha=0.15, color='#1E5FA6')
ax3.set_xlabel('Tempo (s)'); ax3.set_ylabel('w(q)')
ax3.set_title('Manipulabilidade ao Longo da Trajetória')
ax3.legend(); ax3.grid(True, alpha=0.3)

fig.suptitle('Simulação: Trajetória Circular no Plano XY  (DLS, λ=0.05)',
             fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Validação: Jacobiana Numérica vs Analítica

Verificação da implementação: a Jacobiana analítica deve ser igual à Jacobiana calculada por diferenças finitas.

In [ ]:
def jacobian_numerical(q, dh=DH, eps=1e-6):
    """
    Jacobiana numérica de posição (3×n) por diferenças finitas centrais.
    Apenas a parte linear (Jv), para validação.
    """
    n = len(q)
    Jn = np.zeros((3, n))
    for i in range(n):
        dq = np.zeros(n)
        dq[i] = eps
        p_plus  = forward_kinematics(q + dq)[-1][:3, 3]
        p_minus = forward_kinematics(q - dq)[-1][:3, 3]
        Jn[:, i] = (p_plus - p_minus) / (2 * eps)
    return Jn

q_val = np.radians([15, -30, 45, -10, 60, 20])
J_geo  = geometric_jacobian(q_val)[:3, :]   # parte linear
J_num  = jacobian_numerical(q_val)
erro   = np.abs(J_geo - J_num)

print('Jacobiana Geométrica Jv (analítica):')
print(J_geo.round(4))
print('\nJacobiana Numérica Jv (diferenças finitas):')
print(J_num.round(4))
print(f'\nErro máximo |J_analítica - J_numérica|: {erro.max():.2e}  ✓' if erro.max() < 1e-4
      else f'\nErro máximo: {erro.max():.2e}  ✗ verificar implementação')

---
## 9. Exercícios Propostos

1. **Configuração de Singularidade de Ombro**: encontre um $\mathbf{q}$ tal que o efetuador fique sobre o eixo $z_0$ e verifique que $w(\mathbf{q}) = 0$.

2. **Efeito de $\lambda$ na precisão**: implemente um controlador que corrija o erro de posição $e = p_d - p_e$ usando $\dot{\mathbf{x}}_d = K_p \cdot e$. Compare a convergência com $\lambda = 0.001$ vs $\lambda = 0.1$.

3. **Jacobiana Analítica**: implemente a Jacobiana analítica para RPY (roll-pitch-yaw) e compare com a geométrica. Em que configurações elas diferem?

4. **Redundância**: modifique o código para um robô de 7 juntas (adicione um elo extra) e implemente a resolução da redundância pelo espaço nulo:
$$\dot{\mathbf{q}} = J^+ \dot{\mathbf{x}}_d + (I - J^+ J)\,\dot{\mathbf{q}}_0$$

5. **Otimização de Manipulabilidade**: use o espaço nulo para maximizar $w(\mathbf{q})$ ao longo da trajetória:
$$\dot{\mathbf{q}}_0 = k \cdot \nabla_{\mathbf{q}} w(\mathbf{q})$$

In [ ]:
# ---- Espaço de trabalho do exercício 2 (template) ----

def gradient_manipulability(q, dh=DH, eps=1e-4):
    """Gradiente numérico da manipulabilidade em relação a q."""
    grad = np.zeros(len(q))
    for i in range(len(q)):
        dq = np.zeros(len(q)); dq[i] = eps
        grad[i] = (manipulability(q + dq, dh) - manipulability(q - dq, dh)) / (2 * eps)
    return grad

def null_space_projector(J):
    """Projetor no espaço nulo de J: N = I - J+ J"""
    Jp = pseudoinverse(J)
    return np.eye(J.shape[1]) - Jp @ J

# Exemplo: calcular gradiente e projeção
q_ex = np.radians([0, -30, 60, 0, 60, 0])
J_ex = geometric_jacobian(q_ex)
N_ex = null_space_projector(J_ex)
grad_w = gradient_manipulability(q_ex)

print(f'Gradiente de w em q_ex: {grad_w.round(4)}')
print(f'Dimensão do espaço nulo: {np.linalg.matrix_rank(N_ex)} (esperado 0 para robô quadrado não singular)')
print(f'\n→ Para n > 6 (robô redundante), o espaço nulo tem dimensão n-6.')
print(f'  Use N @ grad_w para mover q sem alterar a posição do efetuador.')

---
## Referências

- **Siciliano et al.** — *Robotics: Modelling, Planning and Control* (Springer, 2009) — Cap. 3 e 5  
- **Spong, Hutchinson & Vidyasagar** — *Robot Modeling and Control* (Wiley, 2006) — Cap. 4  
- **Craig, J.J.** — *Introduction to Robotics: Mechanics and Control* (Pearson, 2005) — notação DH padrão  
- **Denso Robotics** — VP-6242 Technical Manual — parâmetros DH e especificações mecânicas  
- **Yoshikawa, T.** — *Manipulability of Robotic Mechanisms* (IJRR, 1985) — índice de manipulabilidade  